In [ ]:
from google.colab import files
import os
import cv2
import numpy as np
import random
from tqdm import tqdm
from PIL import Image

# Upload background images
upload_dir = '/content/backgrounds'
os.makedirs(upload_dir, exist_ok=True)

print("Upload background images")
uploaded = files.upload()
for filename in uploaded:
    os.rename(filename, os.path.join(upload_dir, filename))

# Upload fire PNGs
fire_dir = '/content/fire_pngs'
os.makedirs(fire_dir, exist_ok=True)

print("Upload transparent fire PNG overlays")
uploaded = files.upload()
for filename in uploaded:
    img = Image.open(filename).convert("RGBA")
    save_path = os.path.join(fire_dir, filename)
    img.save(save_path)

# Output folders
output_rgb_dir = '/content/fire_controlnet/rgb'
output_mask_dir = '/content/fire_controlnet/masks'
os.makedirs(output_rgb_dir, exist_ok=True)
os.makedirs(output_mask_dir, exist_ok=True)

def center_crop_and_resize(img, target_dim=(512, 512)):
    h, w = img.shape[:2]
    min_dim = min(h, w)
    top = (h - min_dim) // 2
    left = (w - min_dim) // 2
    crop = img[top:top+min_dim, left:left+min_dim]
    return cv2.resize(crop, target_dim)

def overlay_fire(background_img, fire_dir):
    fire_files = [f for f in os.listdir(fire_dir) if f.lower().endswith('png')]
    if not fire_files:
        raise ValueError("No PNG files found in fire_dir.")
    h_bg, w_bg = background_img.shape[:2]

    # Pick and prep fire overlay
    fire_path = os.path.join(fire_dir, random.choice(fire_files))
    fire_img = Image.open(fire_path).convert("RGBA")

    # Random scale and rotate
    scale = random.uniform(0.1, 0.3)
    fire_w = int(w_bg * scale)
    aspect = fire_img.height / fire_img.width
    fire_h = int(fire_w * aspect)
    fire_img = fire_img.resize((fire_w, fire_h), Image.Resampling.LANCZOS)
    fire_img = fire_img.rotate(random.uniform(-20, 20), expand=True)

    # Make sure it fits
    if fire_img.width > w_bg or fire_img.height > h_bg:
        scale_factor = min(w_bg / fire_img.width, h_bg / fire_img.height) * 0.9
        fire_img = fire_img.resize(
            (max(1, int(fire_img.width * scale_factor)),
             max(1, int(fire_img.height * scale_factor))),
            Image.Resampling.LANCZOS
        )

    #  Placement: keep out of the top 1/3
    bg_top_forbidden = int(h_bg * (1/3))
    x_min = 0
    x_max = max(0, w_bg - fire_img.width)

    # y must be >= bg_top_forbidden
    y_min = min(max(bg_top_forbidden, 0), max(0, h_bg - fire_img.height))
    y_max = max(y_min, h_bg - fire_img.height)

    pos_x = random.randint(x_min, x_max) if x_max >= x_min else 0
    pos_y = random.randint(y_min, y_max) if y_max >= y_min else y_min

    # Paste fire onto background
    bg_pil = Image.fromarray(cv2.cvtColor(background_img, cv2.COLOR_BGR2RGB)).convert("RGBA")
    bg_pil.paste(fire_img, (pos_x, pos_y), fire_img)
    composite = cv2.cvtColor(np.array(bg_pil), cv2.COLOR_RGBA2BGR)

    # Binary mask
    fire_mask = Image.new("L", (w_bg, h_bg), 0)
    fire_alpha = fire_img.split()[-1]
    fire_mask.paste(fire_alpha, (pos_x, pos_y))
    fire_mask_np = np.array(fire_mask)
    fire_mask_np = cv2.threshold(fire_mask_np, 1, 255, cv2.THRESH_BINARY)[1]

    return composite, fire_mask_np

image_files = [f for f in os.listdir(upload_dir) if f.lower().endswith(('png', 'jpg', 'jpeg'))]

for img_file in tqdm(image_files):
    img_path = os.path.join(upload_dir, img_file)
    img = cv2.imread(img_path)
    img = center_crop_and_resize(img)

    fire_img, mask = overlay_fire(img, fire_dir)

    name_base = os.path.splitext(img_file)[0]
    cv2.imwrite(os.path.join(output_rgb_dir, f"rgb_{name_base}.png"), fire_img)
    cv2.imwrite(os.path.join(output_mask_dir, f"mask_{name_base}.png"), mask)

print("Done!")

# Download
import shutil
shutil.make_archive('/content/fire_controlnet_rgb', 'zip', output_rgb_dir)
shutil.make_archive('/content/fire_controlnet_mask', 'zip', output_mask_dir)

from google.colab import files
files.download('/content/fire_controlnet_rgb.zip')
files.download('/content/fire_controlnet_mask.zip')
